# Fast event control : fundamentals

**Level:** 1  
**Prerequisites:** none; entry topic  
**Reading time:** 15 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. A controller receives an event, reduces only observed state, filters registered candidates by their capability contract, then emits an inspectable decision. Fast is an operational budget, not a brain label or a claim that the action is correct.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/episodes/M01-C01-nominal-40000.json`
- `data/derived/episodes/M02-C01-nominal-40000.json`

## Method

The package's guarded FSM is the executable reference for this first run. The canonical C01 trace supplies the exact pre-decision event and candidate set; the notebook reconstructs the public Reflexmesh contracts and asks the released policy to decide again.

\(a_t=\arg\max_{a\in\mathcal A_t^{adm}} U(a\mid o_t)\), with “stop” when no supported action remains. The authorization set is computed independently of learned scores.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M01', 'M02']
sample_case = 'C01'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 2 | case: C01 | seed: 40000
Methods: M01, M02


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M01 | action= a-move | reason= explicit_state_guard | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M02 | action= a-move | reason= reactive_behavior_tree | allowed= ['a-ack_event', 'a-move', 'a-refresh', 'a-inspect', 'a-index', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


## Results

This package replay checks one exact guarded decision on the same frozen observation. It does not demonstrate general transfer, a biological mechanism, or superiority over a matched deployed system.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

An admissible action is a bounded capability, not generated text. The released package reproduces the selected guard on a real frozen trace; that is a mechanics check, not an intelligence claim.

## References

- [JEV](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [LEE](https://doi.org/10.1016/j.neuron.2013.11.028)
- [DPT](https://aclanthology.org/2025.acl-long.206/)